# 📋 Notebook 04 — OCR & Visual Notes Extraction
### AI Meeting Summarization Pipeline

---

## Overview

This notebook extracts **visual notes, slide content, handwriting, and on-screen text**
from meeting and educational videos using **Qwen2.5-VL** — a multimodal vision-language model.

Built for **production use** and fully compatible with the AI Meeting Summarization pipeline.

---

## Full Pipeline

```
Video Input (YouTube URL or Upload)
    ↓
STEP 05 — Video Source Selection
    ↓
STEP 06 — Load Video
    ↓
STEP 07 — Video Metadata & Processing Strategy
    ↓
STEP 08 — Frame Extraction                → frames/
    ↓
STEP 09 — Frame Enhancement               → preprocessed_frames/
    ↓
STEP 10 — Frame Filtering                 → filtered_frames/
    ↓
STEP 11 — Frame Statistics Report
    ↓
STEP 12 — Load Qwen2.5-VL Model
    ↓
STEP 13 — Prompt Construction
    ↓
STEP 14 — Single Frame Analysis           (function definition)
    ↓
STEP 15 — Batch Processing                → frame_results/
    ↓
STEP 16 — Post Processing                 (clean + deduplicate)
    ↓
STEP 17 — Generate Notes JSON             → notes.json
    ↓
STEP 18 — Generate Notes TXT              → notes.txt
    ↓
STEP 19 — Chunk Summaries                 → summaries/chunk_XX.txt
    ↓
STEP 20 — Final Video Summary             → summaries/final_summary.txt
    ↓
STEP 21 — Error Log                       → errors.json
    ↓
STEP 22 — Metadata Report                 → metadata.json
    ↓
STEP 23 — Export Results
```

---

## Output Structure

```
outputs/ocr_notes/<VIDEO_NAME>/
  ├── frames/                   ← raw extracted frames (every 5 sec)
  ├── preprocessed_frames/      ← enhanced frames (CLAHE + sharpen + gamma)
  ├── filtered_frames/          ← frames that passed all quality filters
  ├── chunks/                   ← video chunks for long videos
  ├── frame_results/            ← per-frame JSON output from Qwen
  ├── summaries/
  │   ├── chunk_01_summary.txt
  │   ├── chunk_02_summary.txt
  │   └── final_summary.txt
  ├── notes.json                ← structured pipeline-ready JSON
  ├── notes.txt                 ← human-readable notes
  ├── errors.json               ← error log for failed frames/chunks
  └── metadata.json             ← complete run report
```

---

## Downstream Pipeline Compatibility

Output is structured for use in:
`Transcript Merging → Speaker Diarization → Meeting Summarization →
Decision Extraction → Action Items → RAG Pipeline → Meeting Minutes`


---
## STEP 01 — Install Required Libraries

| Library | Purpose |
|---|---|
| `transformers` / `accelerate` | Qwen2.5-VL model loading and inference |
| `qwen-vl-utils` | Official Qwen vision/video preprocessing |
| `opencv-python` | Frame extraction and image processing |
| `yt-dlp` | YouTube video downloading |
| `Pillow` | Image manipulation and enhancement |
| `imagehash` | Perceptual hashing for duplicate frame detection |
| `tqdm` | Progress bars for long operations |


In [1]:
# Core model dependencies
!pip install -q transformers accelerate

# Qwen official utility package for vision preprocessing
!pip install -q qwen-vl-utils

# Video downloading and image processing tools
!pip install -q yt-dlp opencv-python Pillow imagehash tqdm

print('✅ All libraries installed successfully!')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 20.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 14.7 MB/s eta 0:00:00
✅ All libraries installed successfully!


---
## STEP 02 — Import Required Modules


In [2]:
import os
import cv2
import gc
import json
import math
import shutil
import hashlib
import subprocess
import numpy as np
from pathlib import Path
from datetime import datetime
from difflib import SequenceMatcher

import torch
from PIL import Image, ImageEnhance, ImageFilter
import imagehash
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from tqdm import tqdm

print('✅ All modules imported successfully!')
print(f'   PyTorch  : {torch.__version__}')
print(f'   CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU      : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


✅ All modules imported successfully!
   PyTorch  : 2.11.0+cu128
   CUDA     : True
   GPU      : Tesla T4
   VRAM     : 15.6 GB


---
## STEP 03 — Mount Google Drive


In [3]:
from google.colab import drive

drive.mount('/content/drive')
print('✅ Google Drive mounted successfully!')


Mounted at /content/drive
✅ Google Drive mounted successfully!


---
## STEP 04 — Project Configuration

All pipeline settings are centralized here for easy tuning.

> **VRAM Guide:**
> - **T4  (15 GB)** → `3B` model, `FRAME_FPS=0.2`, `MAX_FRAMES_PER_CHUNK=16`
> - **A100 (40 GB)** → `7B` model, `FRAME_FPS=0.5`, `MAX_FRAMES_PER_CHUNK=32`

| Setting | Default | Description |
|---|---|---|
| `MODEL_ID` | 3B-Instruct | Switch to 7B on A100 |
| `FRAME_FPS` | 0.2 | Frames/sec extracted (= 1 frame every 5 sec) |
| `CHUNK_DURATION_MIN` | 5 | Minutes per summary chunk |
| `MAX_FRAMES_PER_CHUNK` | 16 | Hard cap on frames per Qwen call |
| `BLUR_THRESHOLD` | 80 | Laplacian variance below this = blurry |
| `DARK_THRESHOLD` | 30 | Mean brightness below this = dark/black |
| `BLANK_THRESHOLD` | 8 | Std deviation below this = blank/empty |
| `DUPLICATE_HASH_BITS` | 8 | Perceptual hash size for duplicate detection |
| `SIMILARITY_THRESHOLD` | 0.85 | Fuzzy text similarity for near-duplicate notes |


In [4]:
# ─── Project paths ────────────────────────────────────────────────────────────
PROJECT_FOLDER = '/content/drive/MyDrive/AI_Meeting_Summarization'
OUTPUT_BASE    = f'{PROJECT_FOLDER}/outputs/ocr_notes'

# ─── Model selection ──────────────────────────────────────────────────────────
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'    # use 3B on T4 GPU
# MODEL_ID = 'Qwen/Qwen2.5-VL-7B-Instruct'  # uncomment for A100

# ─── Frame extraction settings ────────────────────────────────────────────────
FRAME_FPS             = 0.2    # 0.2 fps = 1 frame every 5 seconds
SHORT_VIDEO_THRESHOLD = 10     # minutes — below this: no chunking needed
CHUNK_DURATION_MIN    = 5      # minutes per video chunk for summarization

# ─── Qwen inference settings ──────────────────────────────────────────────────
MAX_FRAMES_PER_CHUNK  = 16     # hard cap: max frames sent to Qwen per call
MIN_PIXELS            = 4   * 28 * 28    # safe floor — must be < MAX_PIXELS
MAX_PIXELS            = 256 * 28 * 28    # max frame resolution for Qwen

# ─── Frame quality filter thresholds ──────────────────────────────────────────
BLUR_THRESHOLD        = 80     # Laplacian variance — below = blurry frame
DARK_THRESHOLD        = 30     # mean pixel value — below = dark/black frame
BLANK_THRESHOLD       = 8      # std deviation — below = blank/empty frame
DUPLICATE_HASH_BITS   = 8      # perceptual hash size for duplicate detection
SIMILARITY_THRESHOLD  = 0.85   # fuzzy text similarity for near-duplicate notes

# ─── Prompts by content type ──────────────────────────────────────────────────
PROMPTS = {
    'slide': (
        'هذه صورة لسلايد تعليمي أو عرض تقديمي.\n'
        'المطلوب:\n'
        '1. استخرج جميع النصوص المكتوبة على السلايد كما هي بدون تعديل.\n'
        '2. استخرج العناوين والتعريفات والمعادلات إن وجدت.\n'
        '3. اكتب جملة واحدة تلخص الفكرة الرئيسية.\n'
        'تجاهل: الشعارات، الزخارف، الخلفيات.\n'
        'لا تكرر أي نص. اكتب باللغة العربية.'
    ),
    'whiteboard': (
        'هذه صورة لسبورة بيضاء أو لوح كتابة.\n'
        'المطلوب:\n'
        '1. استخرج جميع النصوص والمعادلات والرسوم البيانية المكتوبة.\n'
        '2. رتب المحتوى بشكل منطقي من الأعلى للأسفل.\n'
        '3. أشر إلى أي رسوم أو أسهم مهمة.\n'
        'اكتب باللغة العربية.'
    ),
    'handwritten': (
        'هذه صورة تحتوي على ملاحظات مكتوبة باليد.\n'
        'المطلوب:\n'
        '1. حاول قراءة وتفسير النص المكتوب حتى لو لم يكن واضحاً 100%.\n'
        '2. اكتب ما تستطيع قراءته بدقة.\n'
        '3. ضع [غير واضح] بجانب الأجزاء التي لا يمكن قراءتها.\n'
        'لا تتجاهل أي كلمة حتى لو كانت صعبة القراءة.\n'
        'اكتب باللغة العربية.'
    ),
    'meeting': (
        'هذه لقطة من اجتماع أو فيديو مسجل.\n'
        'المطلوب:\n'
        '1. استخرج أي نصوص مرئية على الشاشة أو الخلفية.\n'
        '2. اكتب ملاحظة موجزة عما يحدث في هذا المشهد.\n'
        '3. استخرج أي معلومات مهمة مرئية.\n'
        'تجاهل: الوجوه، الملابس، التفاصيل غير ذات الصلة.\n'
        'اكتب باللغة العربية.'
    ),
    'educational': (
        'أنت مساعد متخصص في تحليل الفيديوهات التعليمية العربية.\n'
        'شاهد هذه الصورة بعناية.\n'
        '1. اكتب النصوص المكتوبة على الشاشة كما هي — مرة واحدة فقط.\n'
        '2. استخرج المصطلحات والتعريفات والأمثلة.\n'
        '3. اكتب جملتين تلخصان ما يُشرح.\n'
        'لا تكرر أي كلمة. تجاهل الشعارات والعلامات المائية.\n'
        'اكتب باللغة العربية فقط.'
    ),
}

# Default content type — auto-detected per frame at runtime
DEFAULT_CONTENT_TYPE = 'educational'

# ─── Merge prompt for final summarization ─────────────────────────────────────
MERGE_PROMPT = (
    'فيما يلي ملاحظات مستخرجة من فيديو تعليمي أو اجتماع.\n'
    'اجمعها في ملخص نهائي منظم باللغة العربية يتضمن:\n'
    '- الموضوع الرئيسي\n'
    '- النقاط الأساسية بالترتيب\n'
    '- الأمثلة والمصطلحات المهمة\n'
    '- أي قرارات أو مهام إن وجدت\n'
    'لا تكرر المعلومات. اكتب الملخص النهائي باللغة العربية فقط.\n\n'
    '{notes_content}'
)

print('✅ Configuration loaded successfully!')
print(f'   Model            : {MODEL_ID}')
print(f'   Frame FPS        : {FRAME_FPS} (1 frame every {1/FRAME_FPS:.0f} sec)')
print(f'   Max frames/call  : {MAX_FRAMES_PER_CHUNK}')
print(f'   Chunk duration   : {CHUNK_DURATION_MIN} min')


✅ Configuration loaded successfully!
   Model            : Qwen/Qwen2.5-VL-3B-Instruct
   Frame FPS        : 0.2 (1 frame every 5 sec)
   Max frames/call  : 16
   Chunk duration   : 5 min


---
## STEP 05 — Select Video Source

Two input options are supported:
- **Option 1** → YouTube URL (auto-downloaded with yt-dlp)
- **Option 2** → Upload a local video file directly

Content type is **always auto-detected per frame** — no manual selection needed.


In [5]:
# Display available input options
print('Choose Video Source:')
print('  1 → YouTube URL')
print('  2 → Upload local video')
choice = input('\nEnter choice (1/2): ').strip()

# Get a clean name for this video (used for folder naming and file naming)
VIDEO_NAME = input('\nEnter a name for this video (no spaces): ').strip().replace(' ', '_')

# Content type is always auto-detected per frame — no user input needed
CONTENT_TYPE = 'auto'

print(f'\n✅ Video name     : {VIDEO_NAME}')
print(f'   Source choice  : {choice}')
print(f'   Content type   : auto-detected per frame')


Choose Video Source:
  1 → YouTube URL
  2 → Upload local video

Enter choice (1/2): 1

Enter a name for this video (no spaces): Agentic_AI

✅ Video name     : Agentic_AI
   Source choice  : 1
   Content type   : auto-detected per frame


---
## STEP 06 — Load Video

**YouTube download:**
- Playlist parameters (`&list=`) are stripped automatically
- Timestamp parameters (`&t=`) are stripped automatically
- Cookies file at `MyDrive/cookies.txt` is used if available (bypasses bot detection)
- Falls back to manual upload automatically if download fails

**Manual upload:**
- Accepts any video format supported by OpenCV (MP4, AVI, MOV, etc.)


In [6]:
import yt_dlp
from google.colab import files as colab_files

# ── Initialize error log (collects all errors throughout the pipeline) ─────────
error_log = []

# ── Create full folder structure for this video ───────────────────────────────
VIDEO_FOLDER       = f'{OUTPUT_BASE}/{VIDEO_NAME}'
FRAMES_FOLDER      = f'{VIDEO_FOLDER}/frames'
PREPROCESSED       = f'{VIDEO_FOLDER}/preprocessed_frames'
FILTERED           = f'{VIDEO_FOLDER}/filtered_frames'
CHUNKS_FOLDER      = f'{VIDEO_FOLDER}/chunks'
RESULTS_FOLDER     = f'{VIDEO_FOLDER}/frame_results'
SUMMARIES_FOLDER   = f'{VIDEO_FOLDER}/summaries'

for folder in [VIDEO_FOLDER, FRAMES_FOLDER, PREPROCESSED, FILTERED,
               CHUNKS_FOLDER, RESULTS_FOLDER, SUMMARIES_FOLDER]:
    os.makedirs(folder, exist_ok=True)

VIDEO_PATH = None
VIDEO_URL  = None   # stored for metadata

# ── Option 1: Download from YouTube ───────────────────────────────────────────
if choice == '1':
    VIDEO_URL  = input('Enter YouTube URL: ').strip()

    # Strip playlist parameter — we process single videos only
    if '&list=' in VIDEO_URL:
        VIDEO_URL = VIDEO_URL.split('&list=')[0]
        print(f'🔗 Playlist stripped: {VIDEO_URL}')

    # Strip timestamp parameter — not needed for download
    if '&t=' in VIDEO_URL:
        VIDEO_URL = VIDEO_URL.split('&t=')[0]

    video_file   = f'/content/{VIDEO_NAME}.mp4'
    COOKIES_PATH = '/content/drive/MyDrive/cookies.txt'

    ydl_opts = {
        'format'    : 'best',      # single stream — no merge needed (avoids JS runtime error)
        'outtmpl'   : video_file,
        'noplaylist': True,
        'quiet'     : False,
    }

    # Use cookies if available — bypasses YouTube bot detection on Colab IPs
    if os.path.exists(COOKIES_PATH):
        ydl_opts['cookiefile'] = COOKIES_PATH
        print(f'🍪 Using cookies: {COOKIES_PATH}')
    else:
        print('⚠️  No cookies.txt found — trying without (may fail on some videos).')

    print('\n⬇️  Downloading video from YouTube...')
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([VIDEO_URL])
        VIDEO_PATH = video_file
        print('\n✅ Video downloaded successfully!')
    except Exception as e:
        err_msg = f'YouTube download failed: {str(e)}'
        print(f'\n❌ {err_msg}')
        error_log.append({'step': 'STEP 06', 'type': 'download_error', 'message': err_msg})
        print('\n📤 Falling back to manual upload — please upload the video file:')
        uploaded   = colab_files.upload()
        VIDEO_PATH = f'/content/{list(uploaded.keys())[0]}'
        print(f'\n✅ Video uploaded: {VIDEO_PATH}')

# ── Option 2: Manual upload ────────────────────────────────────────────────────
elif choice == '2':
    print('\n📤 Please upload your video file:')
    uploaded   = colab_files.upload()
    VIDEO_PATH = f'/content/{list(uploaded.keys())[0]}'
    print(f'\n✅ Video uploaded: {VIDEO_PATH}')

else:
    print('❌ Invalid choice. Please re-run this cell and enter 1 or 2.')

print(f'\n📁 Video path: {VIDEO_PATH}')


Enter YouTube URL: https://www.youtube.com/watch?v=XWT6UH0l4Jk
⚠️  No cookies.txt found — trying without (may fail on some videos).

⬇️  Downloading video from YouTube...
[youtube] Extracting URL: https://www.youtube.com/watch?v=XWT6UH0l4Jk
[youtube] XWT6UH0l4Jk: Downloading webpage


[youtube] XWT6UH0l4Jk: Downloading android vr player API JSON
[info] XWT6UH0l4Jk: Downloading 1 format(s): 18
[download] Destination: /content/Agentic_AI.mp4
[download] 100% of   59.27MiB in 00:00:10 at 5.52MiB/s   

✅ Video downloaded successfully!

📁 Video path: /content/Agentic_AI.mp4


---
## STEP 07 — Video Metadata & Processing Strategy

Reads video metadata and decides the processing strategy:

| Video Duration | Strategy |
|---|---|
| ≤ 10 minutes | **Single-pass** — process full video as one unit |
| > 10 minutes | **Chunked** — split into 5-minute chunks, process each independently |

**Metadata collected:**
- Duration (seconds and minutes)
- Original FPS
- Total frame count
- Estimated frames to be extracted at configured FPS


In [7]:
def get_video_info(video_path):
    """Read video duration, FPS, and frame count using OpenCV."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f'Cannot open video: {video_path}')
    fps          = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration_sec = total_frames / fps if fps > 0 else 0
    cap.release()
    return {
        'fps'          : fps,
        'total_frames' : total_frames,
        'duration_sec' : duration_sec,
        'duration_min' : duration_sec / 60,
    }

# Read video metadata
video_info   = get_video_info(VIDEO_PATH)
duration_min = video_info['duration_min']
duration_sec = video_info['duration_sec']

# Estimate how many frames will be extracted at configured FPS
estimated_frames = int(duration_sec * FRAME_FPS)

print('📹 Video Metadata:')
print(f'   Duration            : {duration_min:.1f} min ({duration_sec:.0f} sec)')
print(f'   Original FPS        : {video_info["fps"]:.2f}')
print(f'   Total frames        : {video_info["total_frames"]}')
print(f'   Extraction interval : every {1/FRAME_FPS:.0f} seconds')
print(f'   Estimated extracted : ~{estimated_frames} frames')

# ── Decide processing strategy based on video duration ────────────────────────
if duration_min <= SHORT_VIDEO_THRESHOLD:
    USE_CHUNKING = False
    num_chunks   = 1
    print(f'\n✅ Strategy : SINGLE-PASS (video ≤ {SHORT_VIDEO_THRESHOLD} min)')
else:
    USE_CHUNKING = True
    num_chunks   = math.ceil(duration_min / CHUNK_DURATION_MIN)
    print(f'\n✅ Strategy : CHUNKED PROCESSING')
    print(f'   Chunk size : {CHUNK_DURATION_MIN} minutes')
    print(f'   Chunks     : {num_chunks}')


📹 Video Metadata:
   Duration            : 64.8 min (3888 sec)
   Original FPS        : 30.00
   Total frames        : 116630
   Extraction interval : every 5 seconds
   Estimated extracted : ~777 frames

✅ Strategy : CHUNKED PROCESSING
   Chunk size : 5 minutes
   Chunks     : 13


---
## STEP 08 — Frame Extraction

Extracts one frame every **5 seconds** (at `FRAME_FPS = 0.2`) from the video.
Raw frames are saved as high-quality JPG to `frames/`.

**Frame metadata saved per frame:**
- Frame index number
- Timestamp in seconds
- Timestamp label (MM:SS)
- Source video path

> Enhancement is applied in the **next step** — raw frames are kept untouched here.


In [8]:
def extract_raw_frames(video_path, output_folder, fps=0.2):
    """
    Extract raw frames from a video at the given FPS rate.
    Saves frames as high-quality JPG files.
    Returns list of dicts with frame path and timestamp info.
    """
    cap           = cv2.VideoCapture(video_path)
    video_fps     = cap.get(cv2.CAP_PROP_FPS)
    frame_interval= max(1, int(round(video_fps / fps)))   # extract every Nth frame
    os.makedirs(output_folder, exist_ok=True)

    saved   = []
    idx     = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if idx % frame_interval == 0:
            timestamp_s     = idx / video_fps
            timestamp_label = f'{int(timestamp_s // 60):02d}:{int(timestamp_s % 60):02d}'
            frame_name      = f'frame_{idx:07d}.jpg'
            frame_path      = os.path.join(output_folder, frame_name)

            # Save raw frame at high quality
            cv2.imwrite(frame_path, frame, [cv2.IMWRITE_JPEG_QUALITY, 95])

            saved.append({
                'frame_name'     : frame_name,
                'frame_path'     : frame_path,
                'frame_index'    : idx,
                'timestamp_sec'  : round(timestamp_s, 2),
                'timestamp_label': timestamp_label,
            })

        idx += 1

    cap.release()
    return saved


# ── Extract frames ─────────────────────────────────────────────────────────────
print(f'🎞️  Extracting frames at {FRAME_FPS} FPS (1 frame every {1/FRAME_FPS:.0f} sec)...')

raw_frame_data = extract_raw_frames(VIDEO_PATH, FRAMES_FOLDER, fps=FRAME_FPS)

print(f'\n✅ Extracted {len(raw_frame_data)} raw frames → {FRAMES_FOLDER}')
print(f'   First frame : {raw_frame_data[0]["timestamp_label"] if raw_frame_data else "N/A"}')
print(f'   Last frame  : {raw_frame_data[-1]["timestamp_label"] if raw_frame_data else "N/A"}')


🎞️  Extracting frames at 0.2 FPS (1 frame every 5 sec)...

✅ Extracted 778 raw frames → /content/drive/MyDrive/AI_Meeting_Summarization/outputs/ocr_notes/Agentic_AI/frames
   First frame : 00:00
   Last frame  : 64:45


---
## STEP 09 — Frame Enhancement

Applies a full image enhancement pipeline to every raw frame.
Enhanced frames are saved to `preprocessed_frames/`.

**Enhancement pipeline (applied in order):**
1. **Upscale** — if frame smaller than 224px, upscale with cubic interpolation
2. **CLAHE** — Contrast Limited Adaptive Histogram Equalization on luminance channel only (preserves colors)
3. **Unsharp Mask** — sharpens text edges without halo artifacts
4. **Gamma Correction** — brightens dark frames for better text visibility
5. **FastNlMeans Denoising** — light noise removal without blurring edges
6. **Extra contrast** — applied only for handwritten content type

> CLAHE is applied to the L channel in LAB color space — this is critical for
> Arabic handwriting and whiteboard content with uneven lighting.


In [9]:
def enhance_frame(img_bgr, content_type='educational'):
    """
    Apply full image enhancement pipeline to a single BGR frame.
    Returns enhanced BGR image ready for Qwen processing.
    """
    h, w = img_bgr.shape[:2]

    # ── 1. Upscale small frames (ensures model gets sufficient resolution) ─────
    min_side = 224
    if min(h, w) < min_side:
        scale   = min_side / min(h, w)
        img_bgr = cv2.resize(img_bgr, (int(w * scale), int(h * scale)),
                             interpolation=cv2.INTER_CUBIC)

    # ── 2. CLAHE on luminance channel — improves contrast without color shift ──
    lab     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe   = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l_clahe = clahe.apply(l)
    img_bgr = cv2.cvtColor(cv2.merge([l_clahe, a, b]), cv2.COLOR_LAB2BGR)

    # ── 3. Unsharp mask sharpening — enhances text edges ──────────────────────
    blurred = cv2.GaussianBlur(img_bgr, (0, 0), 3)
    img_bgr = cv2.addWeighted(img_bgr, 1.5, blurred, -0.5, 0)

    # ── 4. Gamma correction — brightens dark frames ───────────────────────────
    mean_brightness = np.mean(img_bgr)
    if mean_brightness < 100:   # apply only to dark frames
        gamma     = 1.5
        lut       = np.array([((i / 255.0) ** (1.0 / gamma)) * 255
                               for i in range(256)], dtype=np.uint8)
        img_bgr   = cv2.LUT(img_bgr, lut)

    # ── 5. Light denoising — removes compression artifacts ────────────────────
    img_bgr = cv2.fastNlMeansDenoisingColored(img_bgr, None, 5, 5, 7, 21)

    # ── 6. Extra contrast boost for handwritten content ───────────────────────
    if content_type == 'handwritten':
        pil_img  = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        enhancer = ImageEnhance.Contrast(pil_img)
        pil_img  = enhancer.enhance(1.8)
        enhancer = ImageEnhance.Sharpness(pil_img)
        pil_img  = enhancer.enhance(2.0)
        img_bgr  = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

    return img_bgr


def enhance_all_frames(raw_frame_data, output_folder):
    """
    Enhance all extracted frames and save to output_folder.
    Returns list of enhanced frame paths (same order as input).
    """
    os.makedirs(output_folder, exist_ok=True)
    enhanced_paths = []

    for frame_info in tqdm(raw_frame_data, desc='Enhancing frames'):
        img = cv2.imread(frame_info['frame_path'])
        if img is None:
            continue

        enhanced = enhance_frame(img, content_type='educational')   # default; refined per-frame later
        out_path = os.path.join(output_folder, frame_info['frame_name'])
        cv2.imwrite(out_path, enhanced, [cv2.IMWRITE_JPEG_QUALITY, 95])
        enhanced_paths.append(out_path)

    return enhanced_paths


print('🎨 Applying enhancement pipeline to all frames...')
print('   (CLAHE + Sharpening + Gamma + Denoising)\n')

enhanced_frame_paths = enhance_all_frames(raw_frame_data, PREPROCESSED)

print(f'\n✅ Enhanced {len(enhanced_frame_paths)} frames → {PREPROCESSED}')
print('✅ Image enhancement pipeline defined!')


🎨 Applying enhancement pipeline to all frames...
   (CLAHE + Sharpening + Gamma + Denoising)



Enhancing frames: 100%|██████████| 778/778 [09:57<00:00,  1.30it/s]


✅ Enhanced 778 frames → /content/drive/MyDrive/AI_Meeting_Summarization/outputs/ocr_notes/Agentic_AI/preprocessed_frames
✅ Image enhancement pipeline defined!


---
## STEP 10 — Frame Filtering

Filters out frames that would waste GPU time on Qwen.
Frames that **pass all filters** are copied to `filtered_frames/`.

| Filter | Method | Threshold | Rationale |
|---|---|---|---|
| **Dark / Black** | Mean pixel value | < 30 | Black transitions, fade-ins |
| **Blank / Empty** | Std deviation | < 8 | Blank slides, solid backgrounds |
| **Blurry** | Laplacian variance | < 80 | Motion blur, out-of-focus frames |
| **Duplicate** | Perceptual hash (pHash) | Hamming ≤ 4 | Static or near-identical frames |

Content type is **auto-detected per frame** for frames that pass all filters.


In [10]:
def is_blurry(img_bgr, threshold=80):
    """Returns True if frame is too blurry to be useful (Laplacian variance check)."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var() < threshold


def is_dark(img_bgr, threshold=30):
    """Returns True if frame is too dark — likely a black screen or transition."""
    return np.mean(img_bgr) < threshold


def is_blank(img_bgr, threshold=8):
    """Returns True if frame has very low variance — nearly blank or solid color."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    return gray.std() < threshold


def get_phash(img_bgr, hash_bits=8):
    """Compute perceptual hash for duplicate detection."""
    pil_img = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    return imagehash.phash(pil_img, hash_size=hash_bits)


def detect_content_type(img_bgr):
    """
    Heuristic content type detection from image characteristics.
    Returns: 'slide', 'whiteboard', 'meeting', or 'educational'
    """
    gray     = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    mean_val = np.mean(gray)
    std_val  = gray.std()

    if mean_val > 200 and std_val < 40:
        return 'whiteboard'       # very bright, very low variance
    elif mean_val > 160 and std_val < 70:
        return 'slide'            # bright slide background
    elif mean_val > 100 and std_val > 60:
        return 'handwritten'      # medium brightness, high texture
    elif mean_val < 80:
        return 'meeting'          # dark frame — screen recording
    else:
        return 'educational'      # default fallback


def filter_frames(frame_paths, content_type='auto',
                  blur_thresh=80, dark_thresh=30,
                  blank_thresh=8, hash_bits=8, hash_dist=4):
    """
    Filter a list of frame paths. Frames that pass all filters are kept.
    Returns (kept_frames_list, filter_stats_dict).

    kept_frames_list: list of dicts with 'path' and 'content_type'
    filter_stats_dict: counts of removed frames per filter type
    """
    kept          = []
    seen_hashes   = []
    stats = {'total': len(frame_paths), 'blurry': 0, 'dark': 0,
             'blank': 0, 'duplicate': 0, 'kept': 0}

    for fpath in tqdm(frame_paths, desc='Filtering frames'):
        img = cv2.imread(fpath)
        if img is None:
            continue

        # Filter 1: dark/black frame (transitions, fade-ins)
        if is_dark(img, dark_thresh):
            stats['dark'] += 1
            continue

        # Filter 2: blank/empty frame (solid backgrounds, blank slides)
        if is_blank(img, blank_thresh):
            stats['blank'] += 1
            continue

        # Filter 3: blurry frame (motion blur, out-of-focus)
        if is_blurry(img, blur_thresh):
            stats['blurry'] += 1
            continue

        # Filter 4: near-duplicate frame (perceptual hash comparison)
        ph     = get_phash(img, hash_bits)
        is_dup = any(abs(ph - prev) <= hash_dist for prev in seen_hashes)
        if is_dup:
            stats['duplicate'] += 1
            continue

        seen_hashes.append(ph)

        # Auto-detect content type per frame
        detected_type = detect_content_type(img) if content_type == 'auto' else content_type
        kept.append({'path': fpath, 'content_type': detected_type})
        stats['kept'] += 1

    return kept, stats


# ── Run frame filtering on enhanced frames ────────────────────────────────────
print('🔍 Filtering enhanced frames...')

filtered_frames, filter_stats = filter_frames(
    frame_paths  = enhanced_frame_paths,
    content_type = CONTENT_TYPE,          # 'auto' — detect per frame
    blur_thresh  = BLUR_THRESHOLD,
    dark_thresh  = DARK_THRESHOLD,
    blank_thresh = BLANK_THRESHOLD,
    hash_bits    = DUPLICATE_HASH_BITS,
)

# Copy filtered frames to filtered_frames/ folder
print(f'\n📂 Copying {filter_stats["kept"]} filtered frames to {FILTERED}...')
for frame_info in filtered_frames:
    dest = os.path.join(FILTERED, os.path.basename(frame_info['path']))
    shutil.copy2(frame_info['path'], dest)
    frame_info['filtered_path'] = dest   # update path to filtered location

print(f'\n✅ Frame filtering complete!')
print('✅ Frame filtering functions defined!')


🔍 Filtering enhanced frames...


Filtering frames: 100%|██████████| 778/778 [00:13<00:00, 59.15it/s]



📂 Copying 235 filtered frames to /content/drive/MyDrive/AI_Meeting_Summarization/outputs/ocr_notes/Agentic_AI/filtered_frames...

✅ Frame filtering complete!
✅ Frame filtering functions defined!


---
## STEP 11 — Frame Statistics Report

Full summary of the frame processing pipeline so far.
Shows exactly how many frames were extracted, enhanced, and filtered.


In [11]:
total_removed = (filter_stats['dark'] + filter_stats['blank'] +
                  filter_stats['blurry'] + filter_stats['duplicate'])
reduction_pct = total_removed / max(filter_stats['total'], 1) * 100

print('=' * 55)
print('📊 FRAME STATISTICS REPORT')
print('=' * 55)
print(f'  Original frames extracted  : {filter_stats["total"]:>6}')
print(f'  Enhanced frames saved      : {len(enhanced_frame_paths):>6}')
print(f'  ─────────────────────────────────────────────────')
print(f'  Removed — dark/black       : {filter_stats["dark"]:>6}')
print(f'  Removed — blank/empty      : {filter_stats["blank"]:>6}')
print(f'  Removed — blurry           : {filter_stats["blurry"]:>6}')
print(f'  Removed — duplicate        : {filter_stats["duplicate"]:>6}')
print(f'  Total removed              : {total_removed:>6}  ({reduction_pct:.1f}%)')
print(f'  ─────────────────────────────────────────────────')
print(f'  ✅ Final frames for Qwen   : {filter_stats["kept"]:>6}')
print('=' * 55)

# Content type breakdown for kept frames
from collections import Counter
ct_counts = Counter(f['content_type'] for f in filtered_frames)
print('\n  Content type breakdown:')
for ct, count in ct_counts.most_common():
    print(f'    {ct:<15} : {count}')


📊 FRAME STATISTICS REPORT
  Original frames extracted  :    778
  Enhanced frames saved      :    778
  ─────────────────────────────────────────────────
  Removed — dark/black       :      0
  Removed — blank/empty      :      1
  Removed — blurry           :      0
  Removed — duplicate        :    542
  Total removed              :    543  (69.8%)
  ─────────────────────────────────────────────────
  ✅ Final frames for Qwen   :    235

  Content type breakdown:
    meeting         : 154
    whiteboard      : 45
    slide           : 19
    handwritten     : 15
    educational     : 2


---
## STEP 12 — Load Qwen2.5-VL Model

Loaded once — reused for all frames throughout the pipeline.

**Memory optimizations applied:**
- `bfloat16` — halves VRAM usage vs float32
- `device_map='auto'` — distributes model across GPU/CPU automatically
- `attn_implementation='eager'` — more stable on T4 GPU (avoids flash-attn issues)
- `PYTORCH_CUDA_ALLOC_CONF=expandable_segments` — reduces memory fragmentation on long runs

> **First run** downloads model weights: ~7 GB (3B) or ~16 GB (7B).
> Subsequent runs load from cache — much faster.


In [12]:
# Reduce CUDA memory fragmentation — important for long videos with many frames
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print(f'⏳ Loading model: {MODEL_ID}')
print('   First run downloads weights — may take a few minutes...\n')

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,       # halves VRAM vs float32
    device_map='auto',                 # auto-distribute across GPU/CPU
    attn_implementation='eager',       # more stable on T4
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=MIN_PIXELS,             # safe floor — must be less than MAX_PIXELS
    max_pixels=MAX_PIXELS,             # hard cap to prevent OOM errors
)

print('\n✅ Model loaded successfully!')
if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'   VRAM used : {used:.1f} GB / {total:.1f} GB')
    print(f'   VRAM free : {total - used:.1f} GB')


⏳ Loading model: Qwen/Qwen2.5-VL-3B-Instruct
   First run downloads weights — may take a few minutes...



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]


✅ Model loaded successfully!
   VRAM used : 7.5 GB / 15.6 GB
   VRAM free : 8.1 GB


---
## STEP 13 — Prompt Construction

Prompts are already defined in **STEP 04** configuration.
This step verifies the prompts and shows a preview of each.

**Prompt selection logic:**
- Content type is auto-detected per frame in STEP 10
- The matching prompt is selected automatically in STEP 14
- No manual prompt selection is needed


In [13]:
print('📝 Available prompts by content type:')
print('=' * 55)
for ct, prompt in PROMPTS.items():
    first_line = prompt.split('\n')[0]
    print(f'  [{ct}] → {first_line}')
print('=' * 55)
print(f'\n✅ {len(PROMPTS)} prompts ready — auto-selected per frame')


📝 Available prompts by content type:
  [slide] → هذه صورة لسلايد تعليمي أو عرض تقديمي.
  [whiteboard] → هذه صورة لسبورة بيضاء أو لوح كتابة.
  [handwritten] → هذه صورة تحتوي على ملاحظات مكتوبة باليد.
  [meeting] → هذه لقطة من اجتماع أو فيديو مسجل.
  [educational] → أنت مساعد متخصص في تحليل الفيديوهات التعليمية العربية.

✅ 5 prompts ready — auto-selected per frame


---
## STEP 14 — Single Frame Analysis Function

Defines the core function that analyzes one frame with Qwen2.5-VL.
This function is called by the batch processor in STEP 15.

**Key inference settings:**
- `max_new_tokens=256` — short focused responses, prevents hallucination loops
- `do_sample=False` — greedy decoding for stable, deterministic output
- `repetition_penalty=1.3` — penalizes repeated tokens at the model level


In [14]:
def analyze_frame_with_qwen(frame_path, content_type='educational'):
    """
    Run Qwen2.5-VL on a single image frame.
    Content type determines which prompt is used.
    Returns the model's text response as a string.
    """
    # Select the appropriate prompt based on content type
    prompt = PROMPTS.get(content_type, PROMPTS['educational'])

    # Build message with image input
    messages = [
        {
            'role': 'user',
            'content': [
                {
                    'type'      : 'image',
                    'image'     : frame_path,    # local file path to the frame
                    'max_pixels': MAX_PIXELS,     # hard cap on resolution
                    'min_pixels': MIN_PIXELS,     # safe floor — must be < MAX_PIXELS
                },
                {
                    'type': 'text',
                    'text': prompt,
                },
            ],
        }
    ]

    # Apply Qwen chat template
    text_input = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # Process vision inputs (loads and preprocesses the image)
    image_inputs, video_inputs = process_vision_info(messages)

    # Tokenize and move to GPU
    inputs = processor(
        text=[text_input],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors='pt',
    ).to(model.device)

    # Run inference
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=256,        # focused output — prevents loops
            do_sample=False,           # greedy decoding — stable and deterministic
            repetition_penalty=1.3,    # penalizes repeated tokens at model level
        )

    # Decode output — trim the input tokens from the output
    trimmed  = [out[len(inp):] for inp, out in zip(inputs.input_ids, output_ids)]
    response = processor.batch_decode(
        trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    return response


def clean_repetitions(text, max_repeat=1):
    """Remove consecutive repeated lines from model output."""
    lines, cleaned, prev, count = text.split('\n'), [], None, 0
    for line in lines:
        stripped = line.strip()
        if stripped == prev:
            count += 1
            if count < max_repeat:
                cleaned.append(line)
        else:
            count = 0
            cleaned.append(line)
            prev = stripped
    return '\n'.join(cleaned)


print('✅ Single frame analysis function defined!')


✅ Single frame analysis function defined!


---
## STEP 15 — Batch Processing

Runs Qwen2.5-VL on all **filtered frames** (from `filtered_frames/`).

**Per-frame JSON saved immediately** to `frame_results/` — so if the notebook
crashes, all processed frames are already saved.

**GPU memory** is released after every frame to prevent OOM on long videos.

> Estimated time on T4: ~5–8 seconds per frame.


In [15]:
all_frame_results = []
frame_note_count  = 0

print(f'🔍 Analyzing {len(filtered_frames)} filtered frames with Qwen2.5-VL...')
print(f'   Estimated time on T4: ~{len(filtered_frames) * 6 // 60} min\n')

for idx, frame_info in enumerate(tqdm(filtered_frames, desc='Analyzing frames')):
    frame_path    = frame_info['path']
    content_type  = frame_info['content_type']
    frame_name    = os.path.basename(frame_path)

    # Estimate timestamp from frame filename
    try:
        frame_num       = int(frame_name.replace('frame_', '').replace('.jpg', ''))
        timestamp_s     = frame_num / video_info['fps']
        timestamp_label = f'{int(timestamp_s // 60):02d}:{int(timestamp_s % 60):02d}'
    except:
        timestamp_s     = 0
        timestamp_label = '00:00'

    try:
        # Run Qwen on this frame
        response  = analyze_frame_with_qwen(frame_path, content_type)

        # Clean repeated lines from output
        response  = clean_repetitions(response)

        # A response with > 20 chars is considered to have real notes
        has_notes = len(response.strip()) > 20

        result = {
            'frame_index'    : idx,
            'frame_file'     : frame_name,
            'timestamp_sec'  : round(timestamp_s, 1),
            'timestamp_label': timestamp_label,
            'content_type'   : content_type,
            'raw_response'   : response,
            'has_notes'      : has_notes,
            'source_model'   : MODEL_ID,
        }

        if has_notes:
            frame_note_count += 1

        # Save per-frame JSON result immediately
        result_path = f'{RESULTS_FOLDER}/{frame_name.replace(".jpg", "_result.json")}'
        with open(result_path, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        all_frame_results.append(result)

    except Exception as e:
        err_msg = f'Frame {frame_name}: {str(e)}'
        print(f'\n❌ Error on frame {frame_name}: {e}')

        # Log error for the errors.json report
        error_log.append({
            'step'       : 'STEP 15',
            'type'       : 'frame_processing_error',
            'frame'      : frame_name,
            'timestamp'  : timestamp_label,
            'message'    : str(e),
        })

        # Store error record to keep track of failed frames
        all_frame_results.append({
            'frame_index'    : idx,
            'frame_file'     : frame_name,
            'timestamp_sec'  : timestamp_s,
            'timestamp_label': timestamp_label,
            'content_type'   : content_type,
            'raw_response'   : f'[ERROR: {str(e)}]',
            'has_notes'      : False,
            'source_model'   : MODEL_ID,
        })

    finally:
        # Always release GPU memory after each frame to prevent OOM
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f'\n✅ Analysis complete!')
print(f'   Frames analyzed         : {len(all_frame_results)}')
print(f'   Frames with real notes  : {frame_note_count}')
print(f'   Frames with errors      : {sum(1 for r in all_frame_results if "ERROR" in r["raw_response"])}')


🔍 Analyzing 235 filtered frames with Qwen2.5-VL...
   Estimated time on T4: ~23 min



Analyzing frames: 100%|██████████| 235/235 [1:02:44<00:00, 16.02s/it]


✅ Analysis complete!
   Frames analyzed         : 235
   Frames with real notes  : 235
   Frames with errors      : 0


---
## STEP 16 — Post Processing

Cleans raw Qwen outputs before saving the final notes files.

**Cleaning operations:**
1. **Remove consecutive repeated lines** — model sometimes loops on static frames
2. **Remove near-duplicate notes** — using fuzzy text similarity (SequenceMatcher ≥ 0.85)
3. **Mark empty outputs** — frames with < 20 character response are flagged as no-content


In [16]:
def remove_repeated_lines(text, max_repeat=1):
    """Remove consecutive repeated lines from model output."""
    lines, cleaned, prev, count = text.split('\n'), [], None, 0
    for line in lines:
        s = line.strip()
        if s == prev:
            count += 1
            if count < max_repeat:
                cleaned.append(line)
        else:
            count = 0
            cleaned.append(line)
            prev = s
    return '\n'.join(cleaned)


def is_near_duplicate_text(text_a, text_b, threshold=0.85):
    """True if two texts are highly similar — likely duplicate notes."""
    return SequenceMatcher(None, text_a, text_b).ratio() > threshold


def post_process_results(results, similarity_threshold=0.85):
    """
    Clean all frame results:
    - Remove repeated lines within each result
    - Mark near-duplicate consecutive frames as skipped
    Returns cleaned results list.
    """
    cleaned    = []
    prev_notes = ''
    dup_count  = 0

    for result in results:
        if not result.get('has_notes'):
            cleaned.append(result)
            continue

        # Clean repeated lines in this frame's output
        result['raw_response'] = remove_repeated_lines(result['raw_response'])

        # Skip if this frame's notes are too similar to the previous frame
        if is_near_duplicate_text(result['raw_response'], prev_notes, similarity_threshold):
            result['has_notes']    = False
            result['raw_response'] = '[NEAR-DUPLICATE — skipped in post-processing]'
            dup_count             += 1
            cleaned.append(result)
            continue

        prev_notes = result['raw_response']
        cleaned.append(result)

    return cleaned, dup_count


# ── Apply post-processing ──────────────────────────────────────────────────────
print('🧹 Post-processing results...')
all_frame_results, post_dup_count = post_process_results(
    all_frame_results, SIMILARITY_THRESHOLD
)

# Recount after post-processing
frame_note_count = sum(1 for r in all_frame_results if r.get('has_notes'))

print(f'\n✅ Post-processing complete!')
print(f'   Near-duplicates removed  : {post_dup_count}')
print(f'   Final frames with notes  : {frame_note_count}')


🧹 Post-processing results...

✅ Post-processing complete!
   Near-duplicates removed  : 0
   Final frames with notes  : 235


---
## STEP 17 — Generate Notes JSON

Saves all results in a structured JSON file — the **primary output** for the pipeline.

This JSON is the interface between this notebook and all downstream notebooks:
- Transcript Merging
- RAG Pipeline
- Meeting Minutes Generation
- Decision / Action Item Extraction
- Speaker Diarization alignment

**Per-frame record structure:**
```json
{
  "frame_file": "frame_0000300.jpg",
  "timestamp_sec": 10.0,
  "timestamp_label": "00:10",
  "content_type": "slide",
  "raw_response": "...",
  "has_notes": true,
  "source_model": "Qwen/Qwen2.5-VL-3B-Instruct"
}
```


In [17]:
# ── Build pipeline-ready JSON ─────────────────────────────────────────────────
notes_json = {
    'video_name'     : VIDEO_NAME,
    'video_path'     : VIDEO_PATH,
    'duration_min'   : round(video_info['duration_min'], 2),
    'content_type'   : CONTENT_TYPE,
    'model_id'       : MODEL_ID,
    'processed_at'   : str(datetime.now()),
    'pipeline_ready' : True,
    'quality_report' : {
        'total_frames_extracted' : filter_stats['total'],
        'frames_removed_dark'    : filter_stats['dark'],
        'frames_removed_blank'   : filter_stats['blank'],
        'frames_removed_blurry'  : filter_stats['blurry'],
        'frames_removed_dupes'   : filter_stats['duplicate'],
        'frames_analyzed'        : filter_stats['kept'],
        'frames_with_notes'      : frame_note_count,
        'frames_with_errors'     : sum(1 for r in all_frame_results if 'ERROR' in r['raw_response']),
    },
    'frames': all_frame_results,
}

# Save structured JSON
notes_json_path = f'{VIDEO_FOLDER}/{VIDEO_NAME}_notes.json'
with open(notes_json_path, 'w', encoding='utf-8') as f:
    json.dump(notes_json, f, ensure_ascii=False, indent=2)

print(f'✅ Structured notes JSON saved: {notes_json_path}')
print(f'   Total frames in JSON : {len(notes_json["frames"])}')
print(f'   Frames with notes    : {frame_note_count}')


✅ Structured notes JSON saved: /content/drive/MyDrive/AI_Meeting_Summarization/outputs/ocr_notes/Agentic_AI/Agentic_AI_notes.json
   Total frames in JSON : 235
   Frames with notes    : 235


---
## STEP 18 — Generate Notes TXT

Saves a clean, human-readable notes file ordered by timestamp.
Only frames that produced meaningful content (`has_notes = True`) are included.

**Format:**
```
[00:10]  frame_0000300.jpg  (slide)
────────────────────────────────────────────────────────────
Extracted text and summary here...
```


In [18]:
notes_txt_path = f'{VIDEO_FOLDER}/{VIDEO_NAME}_notes.txt'

with open(notes_txt_path, 'w', encoding='utf-8') as f:
    f.write(f'VISUAL NOTES EXTRACTION REPORT\n')
    f.write(f'Video   : {VIDEO_NAME}\n')
    f.write(f'Date    : {datetime.now().strftime("%Y-%m-%d %H:%M")}\n')
    f.write(f'Model   : {MODEL_ID}\n')
    f.write(f'Frames  : {frame_note_count} frames with notes / '
            f'{filter_stats["kept"]} analyzed\n')
    f.write('=' * 60 + '\n\n')

    for result in all_frame_results:
        if not result.get('has_notes'):
            continue
        f.write(f'[{result["timestamp_label"]}]  '
                f'{result["frame_file"]}  '
                f'({result["content_type"]})\n')
        f.write('-' * 60 + '\n')
        f.write(result['raw_response'])
        f.write('\n\n')

print(f'✅ Notes TXT saved: {notes_txt_path}')


✅ Notes TXT saved: /content/drive/MyDrive/AI_Meeting_Summarization/outputs/ocr_notes/Agentic_AI/Agentic_AI_notes.txt


---
## STEP 19 — Chunk Summaries

Divides all extracted notes into **5-minute time windows** and generates
a focused summary for each chunk using a text-only Qwen call.

This is much faster than frame analysis — no images are sent to the model.

Each chunk summary is saved to:
`summaries/chunk_01_summary.txt`, `chunk_02_summary.txt`, etc.


In [19]:
def summarize_chunk(chunk_notes_text, chunk_num, chunk_label):
    """
    Generate a summary for one time-window chunk.
    Uses text-only Qwen call — no image input needed.
    Returns (summary_text, saved_file_path).
    """
    prompt = (
        'فيما يلي ملاحظات مستخرجة من مقطع فيديو تعليمي.\n'
        'اكتب ملخصاً موجزاً باللغة العربية يتضمن:\n'
        '- الموضوع الرئيسي في هذا المقطع\n'
        '- النقاط الأساسية والمصطلحات\n'
        'لا تكرر المعلومات. أجب باللغة العربية فقط.\n\n'
        f'{chunk_notes_text}'
    )

    messages   = [{'role': 'user', 'content': [{'type': 'text', 'text': prompt}]}]
    text_input = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = processor(text=[text_input], return_tensors='pt').to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=512,
            do_sample=False, repetition_penalty=1.2,
        )

    trimmed  = [out[len(inp):] for inp, out in zip(inputs.input_ids, output_ids)]
    summary  = processor.batch_decode(
        trimmed, skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    # Save chunk summary to summaries folder
    chunk_path = f'{SUMMARIES_FOLDER}/chunk_{chunk_num:02d}_summary.txt'
    with open(chunk_path, 'w', encoding='utf-8') as f:
        f.write(f'CHUNK {chunk_num:02d} SUMMARY  [{chunk_label}]\n')
        f.write('=' * 40 + '\n\n')
        f.write(summary)

    return summary, chunk_path


# ── Build time-window chunks and generate summaries ───────────────────────────
chunk_sec         = CHUNK_DURATION_MIN * 60
useful_results    = [r for r in all_frame_results if r.get('has_notes')]
chunk_summaries   = []

if useful_results:
    max_ts     = max(r['timestamp_sec'] for r in useful_results)
    num_chunks = max(1, math.ceil(max_ts / chunk_sec))

    print(f'📝 Generating {num_chunks} chunk summaries ({CHUNK_DURATION_MIN} min each)...\n')

    for chunk_idx in range(num_chunks):
        start_s     = chunk_idx * chunk_sec
        end_s       = start_s + chunk_sec
        chunk_label = (f'{int(start_s//60):02d}:00 – '
                       f'{int(end_s//60):02d}:00')

        # Collect notes from frames within this time window
        chunk_frames = [
            r for r in useful_results
            if start_s <= r['timestamp_sec'] < end_s
        ]

        if not chunk_frames:
            print(f'   ⏩ Chunk {chunk_idx+1:02d} [{chunk_label}]: no content — skipping')
            continue

        chunk_text = '\n\n'.join(
            f'[{r["timestamp_label"]}] ({r["content_type"]})\n{r["raw_response"]}'
            for r in chunk_frames
        )

        try:
            summary, path = summarize_chunk(chunk_text, chunk_idx + 1, chunk_label)
            chunk_summaries.append({
                'chunk'  : chunk_idx + 1,
                'label'  : chunk_label,
                'summary': summary,
                'file'   : path,
            })
            print(f'   ✅ Chunk {chunk_idx+1:02d} [{chunk_label}] saved')
        except Exception as e:
            err_msg = f'Chunk {chunk_idx+1} summary failed: {str(e)}'
            print(f'   ❌ {err_msg}')
            error_log.append({'step': 'STEP 19', 'type': 'chunk_summary_error',
                               'chunk': chunk_idx+1, 'message': str(e)})

        # Release GPU memory after each chunk summary
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(f'\n✅ {len(chunk_summaries)} chunk summaries generated!')
else:
    print('⚠️  No useful frames — skipping chunk summaries.')


📝 Generating 13 chunk summaries (5 min each)...

   ✅ Chunk 01 [00:00 – 05:00] saved
   ✅ Chunk 02 [05:00 – 10:00] saved
   ✅ Chunk 03 [10:00 – 15:00] saved
   ✅ Chunk 04 [15:00 – 20:00] saved
   ❌ Chunk 5 summary failed: CUDA out of memory. Tried to allocate 3.15 GiB. GPU 0 has a total capacity of 14.56 GiB of which 2.39 GiB is free. Including non-PyTorch memory, this process has 12.17 GiB memory in use. Of the allocated memory 11.98 GiB is allocated by PyTorch, and 68.56 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)
   ❌ Chunk 6 summary failed: CUDA out of memory. Tried to allocate 2.44 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.06 GiB is free. Including non-PyTorch memory, this process has 13.50 Gi

---
## STEP 20 — Final Video Summary

Merges all chunk summaries into one comprehensive final summary.
Uses a text-only Qwen call — fast, no image input needed.

Saved to `summaries/final_summary.txt`.


In [22]:
def generate_final_summary(chunk_summaries, merge_prompt_template, model, processor):
    """
    Merge all chunk summaries into one final summary using text-only Qwen call.
    """
    combined_summaries = ''
    for chunk_info in chunk_summaries:
        combined_summaries += (
            f'[{chunk_info["label"]}]\n'
            f'{chunk_info["summary"]}\n\n'
        )

    if not combined_summaries.strip():
        return 'لم يتم استخراج ملاحظات كافية من هذا الفيديو.'

    merge_prompt = merge_prompt_template.format(notes_content=combined_summaries)

    messages = [
        {'role': 'user', 'content': [{'type': 'text', 'text': merge_prompt}]}
    ]

    text_input = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )

    inputs = processor(text=[text_input], return_tensors='pt').to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=False,
            repetition_penalty=1.2,
        )

    trimmed  = [out[len(inp):] for inp, out in zip(inputs.input_ids, output_ids)]
    response = processor.batch_decode(
        trimmed, skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return response.strip()


print('🧠 Generating final summary...\n')

final_summary = generate_final_summary(
    chunk_summaries, MERGE_PROMPT, model, processor
)

# Save final summary
summary_path = f'{SUMMARIES_FOLDER}/final_summary.txt'
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(f'FINAL VISUAL NOTES SUMMARY\n')
    f.write(f'Video : {VIDEO_NAME}\n')
    f.write(f'Date  : {datetime.now().strftime("%Y-%m-%d %H:%M")}\n')
    f.write('=' * 60 + '\n\n')
    f.write(final_summary)

print(f'✅ Summary saved: {summary_path}')
print('\n' + '─' * 60)
print(final_summary[:800])

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

🧠 Generating final summary...

✅ Summary saved: /content/drive/MyDrive/AI_Meeting_Summarization/outputs/ocr_notes/Agentic_AI/summaries/final_summary.txt

────────────────────────────────────────────────────────────
إليك الملخص النهائي المنظم والموسع لل录像 التعليمي/التواصل الذي تم تنقشه:

---

## ملخص موجز للمقطع التعليمي

**موضوع الرئيسي:** بناء أول مساعد ذكي (AI).

**نقاط أساسية:**

1. **الأهداف**: بناء أول مساعد ذكي قادر على التعامل مع مجموعة متنوعة من الأسئلة والأحداث.
2. **عمليات الأساس**: فهم العمليات البسيطَة التي يقوم بها الإنسان عند الإجابة عن أسئله وأداء عمله اليومي.
3. **تقنيات التوجيه**: استخدام التوجيه القواعد لتحكم الـ LLM باستخدام الشخصيات المستخدمة كمساعدين ذكية للذكاء الاصطناعي.
4. **تحديد المدخلات الصحيحة**: اتباع تقنيات التوجيه القواعد لتحويل الوظائف التالية بشكل مثل الأجهزة الذكية للتعلم الآلي.
5. **استخدام أدوات البحث الإحترافية**: استغلال أدوات البحث الإحترافية لإنشاء برامج أكثر فائدة وإنتاجية.
6. **تجنب الأخطاء الشائعة**: اتخاذ خطوات لتجنب الأخطاء الشائعة أثناء تصم

---
## STEP 21 — Error Log

Saves all errors collected throughout the pipeline to `errors.json`.

Errors are logged (not raised) throughout the pipeline so processing continues
even when individual frames or chunks fail.

**Error log structure:**
```json
{
  "step": "STEP 15",
  "type": "frame_processing_error",
  "frame": "frame_0001200.jpg",
  "timestamp": "01:20",
  "message": "CUDA out of memory..."
}
```


In [23]:
errors_path = f'{VIDEO_FOLDER}/errors.json'

errors_report = {
    'video_name'  : VIDEO_NAME,
    'created_at'  : str(datetime.now()),
    'total_errors': len(error_log),
    'errors'      : error_log,
}

with open(errors_path, 'w', encoding='utf-8') as f:
    json.dump(errors_report, f, ensure_ascii=False, indent=2)

if error_log:
    print(f'⚠️  {len(error_log)} errors logged → {errors_path}')
    for err in error_log:
        print(f'   [{err["step"]}] {err["type"]}: {err["message"][:80]}')
else:
    print(f'✅ No errors! Clean run → {errors_path}')


⚠️  4 errors logged → /content/drive/MyDrive/AI_Meeting_Summarization/outputs/ocr_notes/Agentic_AI/errors.json
   [STEP 19] chunk_summary_error: CUDA out of memory. Tried to allocate 3.15 GiB. GPU 0 has a total capacity of 14
   [STEP 19] chunk_summary_error: CUDA out of memory. Tried to allocate 2.44 GiB. GPU 0 has a total capacity of 14
   [STEP 19] chunk_summary_error: CUDA out of memory. Tried to allocate 4.22 GiB. GPU 0 has a total capacity of 14
   [STEP 19] chunk_summary_error: CUDA out of memory. Tried to allocate 3.06 GiB. GPU 0 has a total capacity of 14


---
## STEP 22 — Metadata Report

Saves a complete, production-level run report to `metadata.json`.

Includes all fields needed for:
- **Debugging** — exact settings and counts for every processing stage
- **Experiment tracking** — model ID, thresholds, pixel settings
- **Reproducibility** — full configuration snapshot
- **Pipeline handoff** — paths to all output files for downstream notebooks


In [24]:
metadata = {
    # ── Video info ────────────────────────────────────────────────────────────
    'video_name'                   : VIDEO_NAME,
    'video_source'                 : 'youtube' if choice == '1' else 'upload',
    'video_url'                    : VIDEO_URL,
    'video_path'                   : VIDEO_PATH,
    'duration_seconds'             : round(video_info['duration_sec'], 1),
    'duration_minutes'             : round(video_info['duration_min'], 2),
    'fps'                          : video_info['fps'],
    'total_frames'                 : video_info['total_frames'],

    # ── Extraction settings ───────────────────────────────────────────────────
    'frame_extraction_interval_sec': round(1 / FRAME_FPS, 1),
    'extracted_frames'             : filter_stats['total'],
    'preprocessed_frames'          : len(enhanced_frame_paths),

    # ── Filter results ────────────────────────────────────────────────────────
    'duplicate_frames_removed'     : filter_stats['duplicate'],
    'blurry_frames_removed'        : filter_stats['blurry'],
    'dark_frames_removed'          : filter_stats['dark'],
    'empty_frames_removed'         : filter_stats['blank'],
    'final_frames_for_qwen'        : filter_stats['kept'],

    # ── Model settings ────────────────────────────────────────────────────────
    'model_id'                     : MODEL_ID,
    'min_pixels'                   : MIN_PIXELS,
    'max_pixels'                   : MAX_PIXELS,
    'max_frames_per_chunk'         : MAX_FRAMES_PER_CHUNK,

    # ── Processing strategy ───────────────────────────────────────────────────
    'processing_strategy'          : 'chunked' if USE_CHUNKING else 'single_pass',
    'chunk_duration_min'           : CHUNK_DURATION_MIN,
    'num_chunks'                   : num_chunks,
    'chunks_processed'             : len(chunk_summaries),
    'chunks_with_errors'           : sum(1 for e in error_log if e['step'] == 'STEP 19'),

    # ── Output quality ────────────────────────────────────────────────────────
    'notes_extracted'              : frame_note_count,
    'summary_generated'            : True,
    'total_errors'                 : len(error_log),

    # ── Output paths ──────────────────────────────────────────────────────────
    'output_folder'                : VIDEO_FOLDER,
    'frames_folder'                : FRAMES_FOLDER,
    'preprocessed_frames_folder'   : PREPROCESSED,
    'filtered_frames_folder'       : FILTERED,
    'ocr_json_file'                : notes_json_path,
    'ocr_txt_file'                 : notes_txt_path,
    'summary_file'                 : summary_path,
    'errors_file'                  : errors_path,

    # ── Run info ──────────────────────────────────────────────────────────────
    'pipeline_ready'               : True,
    'created_at'                   : str(datetime.now()),
}

metadata_path = f'{VIDEO_FOLDER}/metadata.json'
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print('✅ Metadata saved:', metadata_path)
print()
print('📊 FINAL PIPELINE REPORT')
print('=' * 55)
print(f'  Video duration       : {metadata["duration_minutes"]:.1f} min')
print(f'  Frames extracted     : {metadata["extracted_frames"]}')
print(f'  Frames for Qwen      : {metadata["final_frames_for_qwen"]}')
print(f'  Notes extracted      : {metadata["notes_extracted"]}')
print(f'  Chunk summaries      : {metadata["chunks_processed"]}')
print(f'  Errors               : {metadata["total_errors"]}')
print('=' * 55)
print(f'\n📁 All outputs saved in:\n   {VIDEO_FOLDER}')

✅ Metadata saved: /content/drive/MyDrive/AI_Meeting_Summarization/outputs/ocr_notes/Agentic_AI/metadata.json

📊 FINAL PIPELINE REPORT
  Video duration       : 64.8 min
  Frames extracted     : 778
  Frames for Qwen      : 235
  Notes extracted      : 235
  Chunk summaries      : 9
  Errors               : 4

📁 All outputs saved in:
   /content/drive/MyDrive/AI_Meeting_Summarization/outputs/ocr_notes/Agentic_AI


---
## STEP 23 — Export Results

Downloads the main output files directly to your computer.


In [25]:
from google.colab import files

print('⬇️  Downloading results...')
files.download(summary_path)
files.download(notes_txt_path)
files.download(notes_json_path)
files.download(metadata_path)
print('✅ All files downloaded!')

⬇️  Downloading results...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ All files downloaded!
